# W04A · 이웃을 고려한 CF와 사용자의 평가 경향
**딥러닝응용I(추천시스템) · 동덕여자대학교 데이터사이언스전공 · 유원상 교수 · 2026-2**

2026-09-22 · 주교재3.4–3.6. 목표 영화 평가자 중 이웃을 고르고, 검증으로 k를 선택하고, 사용자 평균을 보정한다.
[강의자료](https://github.com/lunalab-ai/recommender/blob/2026-fall-w04a/course/notion/sessions/w04a-neighborhood-cf.md)

CPU에서 첫 셀부터 실행한다. `None` 빈칸은 미완성이어도 독립 시연을 막지 않는다. E1–E6은 예상·구현·비교 활동이다.


## 1. 실행 환경과 작은 평점표
고정 수업 태그의 패키지를 현재 커널에 설치한다. 설치 오류는 숨기지 않는다. 런타임 재시작 안내가 나오면 재시작 후 첫 셀부터 실행한다.
`toy_ratings()`는 5명·5편의 독자 합성 관측18행을 만든다. `NeighborCF(k,centered,threshold)`는 설정을 저장하고 `fit(train)`이 관측 행렬·코사인·평균을 만든다. k=0은 전체 이웃이다.
[모델/API 정의](https://github.com/lunalab-ai/recommender/blob/2026-fall-w04a/src/luna_recsys/neighborhood.py)


In [ ]:
import os, sys, subprocess
os.environ['GRADIO_ANALYTICS_ENABLED']='False'
IN_COLAB='google.colab' in sys.modules
COURSE_VERSION='2026-fall-w04a'
if IN_COLAB:
    subprocess.run([sys.executable,'-m','pip','install','-q',
        f'luna-recommender-course[apps] @ git+https://github.com/lunalab-ai/recommender.git@{COURSE_VERSION}',
        'gradio==6.27.0'],check=True)
import numpy as np
import pandas as pd
import gradio as gr
from IPython.display import display
from sklearn.metrics import root_mean_squared_error
from luna_recsys.collaborative import UserCF, toy_ratings
from luna_recsys.neighborhood import NeighborCF, validation_sweep
from luna_recsys.neighborhood_app import neighbor_view, build_neighbor_app
from luna_recsys.cf_app import build_cf_app
from luna_recsys.ranking import FourMethodRecommender
from luna_recsys.datasets import prepare_movielens
from luna_recsys.evaluation import split_ratings
toy=toy_ratings()
small=NeighborCF(k=2).fit(toy)
R=small.rating_matrix_
display(R)
print('Python',sys.version.split()[0],'NumPy',np.__version__,'pandas',pd.__version__,'Gradio',gr.__version__)


### E1 · 목표 평가자부터 고르기
U1의 영화 E(ID5)를 k=1로 예측한다. 아래 함수는 `R`, 유사도Series, 대상ID, 영화ID, k를 받아 선택한 이웃ID 목록을 반환해야 한다. 목표 영화 평가자와 양의 유사도를 먼저 고르고, 자신을 제외한 뒤 정렬하라. 전체 사용자 상위 k명을 먼저 고르면 어떤 문제가 생기는지도 적어라.
pandas의 `notna()`는 관측 여부, `sort_values()`는 값 정렬, `head(k)`는 앞 k행 선택이다.


In [ ]:
def choose_neighbors(rating_matrix, similarities, user_id, movie_id, k):
    return None  # 선택된 사용자 ID 목록을 반환하도록 완성
chosen=choose_neighbors(R,small.similarity_.loc[1],1,5,1)
if chosen is None:
    print('E1: 목표 영화 평가자 안에서 이웃을 선택하세요.')
else:
    assert list(chosen)==small.explain(1,5).head(1).user_id.tolist()


## 2. 선택된 이웃과 가중평균
`explain(u,i)`는 선택된 전체 이웃의 유사도·평점·사용자 평균·편차·정규화 가중치·기여도를 반환한다. 원평점 방식의 기여도합은 clip 전 예측과 같다. `predict_details` 입력의 rating 열은 예측에 사용하지 않는다.


In [ ]:
pair=pd.DataFrame({'user_id':[1],'movie_id':[4]})
display(small.explain(1,4))
display(small.predict_details(pair))


### E2 · 유사도와 평점의 대응 오류
버그: `s.sort_values(ascending=False).to_numpy() @ ratings.to_numpy()`처럼 유사도만 정렬하면 사용자 대응이 깨진다. D의 후보를 사용자ID로 결합한 DataFrame에서 함께 정렬하여 k=2 원평점 예측을 계산하라. `pd.concat(...,axis=1)`은 인덱스로 열을 정렬해 결합한다.


In [ ]:
aligned_prediction=None
if aligned_prediction is None:
    print('E2: 사용자ID를 유지한 표 전체를 정렬하세요.')
else:
    assert np.isclose(aligned_prediction,small.predict(pair)[0])


## 3. MovieLens 준비와 관측 분할
공식 MovieLens100K를 자동 다운로드하고 크기·ID를 검사한다. 다음 실행은 캐시를 사용한다. 기본 `real` 실패는 오류로 알린다. 네트워크가 불가능한 경우에만 `DATA_MODE='synthetic'`을 직접 선택한다. 합성 결과는 강의의 실제 측정값과 다르다.
`split_ratings`는 가능하면 사용자별로 층화한다. 바깥20%를 test로 남기고 남은80%의25%를 validation으로 분리하면60/20/20이다. 같은 사용자와 영화가 여러 분할에 있어도 같은 관측 쌍은 겹치면 안 된다.
[데이터 준비 API](https://github.com/lunalab-ai/recommender/blob/2026-fall-w04a/src/luna_recsys/datasets.py) · [분할 API](https://github.com/lunalab-ai/recommender/blob/2026-fall-w04a/src/luna_recsys/evaluation.py)


In [ ]:
DATA_MODE='real'
prepared=prepare_movielens('data/local/w04a',mode=DATA_MODE)
data=prepared.data
print(prepared.description)
print('사용자/영화/평점:',len(data.users),len(data.movies),len(data.ratings))
outer=split_ratings(data.ratings,test_size=.2,seed=20260922)
inner=split_ratings(outer.train,test_size=.25,seed=20260923)
train,validation,test=inner.train,inner.test,outer.test
print('train/validation/test:',len(train),len(validation),len(test))
def pair_set(frame):
    return set(frame[['user_id','movie_id']].itertuples(index=False,name=None))
assert not pair_set(train)&pair_set(validation)
assert not pair_set(train)&pair_set(test)
assert not pair_set(validation)&pair_set(test)


### 같은 검증 관측에서 이웃 수 비교
`validation_sweep(train,validation,k_values)`는 train에서만 통계를 계산하고16개 후보의 검증 RMSE와 coverage, 평균 실제 이웃 수를 반환한다. 이 함수에는 test 인자가 없다. k=0은 모든 양의 유사도 평가자다. 표는 RMSE 오름차순이며 동점은 k와 보정 여부 순이다.


In [ ]:
scores=validation_sweep(train,validation,[1,5,10,20,30,40,60,0])
display(scores.round(6))
display(scores.pivot(index='k',columns='centered',values='rmse').rename(columns={False:'원평점',True:'평균 보정'}))


### E3 · 선택용 데이터와 마지막 평가
표에서 선택할 k와 보정 여부를 적어라. test RMSE로16개 후보를 정렬하는 코드는 왜 잘못된 선택 절차인지 설명하라. 검증 선택 후 train+validation으로 다시 학습해도 되는 이유와 그때 test를 합치면 안 되는 이유도 적어라.


In [ ]:
selected_k=None
selected_centered=None
if selected_k is None or selected_centered is None:
    print('E3: validation 표에서 설정을 선택하세요.')
else:
    assert int(selected_k)==int(scores.iloc[0].k)
    assert bool(selected_centered)==bool(scores.iloc[0].centered)


### 고정한 설정으로 test 한 번 평가
다음은 빈칸과 독립된 시연이다. 검증표의 최저 설정을 고정하고80% 관측으로 재학습한다. 최종 test 정답은 예측을 만든 뒤 RMSE 계산에만 전달한다. 작은 RMSE는 순위 만족도 보장이 아니다.


In [ ]:
best=scores.iloc[0]
final_model=NeighborCF(k=int(best.k),centered=bool(best.centered)).fit(outer.train)
final_details=final_model.predict_details(test[['user_id','movie_id']])
final_rmse=root_mean_squared_error(test.rating,final_details.prediction)
print('고정한 설정:',int(best.k),bool(best.centered))
print('test RMSE:',final_rmse,'CF coverage:',final_details.basis.eq('cf').mean())
print('clip 적용:',int((final_details.prediction!=final_details.raw_prediction).sum()))


## 4. 사용자의 평가 경향 보정
$$\widehat r_{ui}=\overline r_u+\frac{\sum_{v\in N_{u,i}}s_{uv}(r_{vi}-\overline r_v)}{\sum_{v\in N_{u,i}}s_{uv}}.$$
이웃 평균을 빼 상대적 선호를 모으고 대상 사용자 평균을 더한다. 원평점 코사인은 그대로 유지한다. 평균은 train 관측만 사용한다. `raw_prediction`은 clip 전, `prediction`은1~5점 범위의 최종값이다.


In [ ]:
centered_small=NeighborCF(k=2,centered=True).fit(toy)
display(centered_small.user_means_.rename('관측 사용자 평균'))
display(centered_small.explain(1,4))
display(centered_small.predict_details(pair))


### E4 · 누구의 평균을 빼고 더하는가?
버그① `R.fillna(0).mean(axis=0)`으로 사용자 평균을 계산했다. 버그② 모든 이웃 평점에서 U1 평균을 빼고 마지막에 이웃 평균을 더했다. 두 오류를 설명하고 U1–D(k2) 보정 예측을 직접 계산하라. `axis=1`은 사용자별 행 평균이다.


In [ ]:
corrected_prediction=None
if corrected_prediction is None:
    print('E4: 관측 행 평균 → 각 이웃 평균 제거 → 대상 평균 복원')
else:
    assert np.isclose(corrected_prediction,centered_small.predict(pair)[0])


## 5. 누적 추천 앱의 callback
앱은 설정 효과를 탐색하는 train 모델을 사용한다. 최종 test 점수 선택과 분리한다. `neighbor_view(model,movies,user_id,k,centered,threshold,top_n)`은 요약HTML·추천표·첫 추천의 전체 이웃 표를 반환한다. 원본 모델 설정은 변경하지 않는다. `top_n`은 추천 영화 수다.
[callback/API 정의](https://github.com/lunalab-ai/recommender/blob/2026-fall-w04a/src/luna_recsys/neighborhood_app.py)


In [ ]:
app_model=NeighborCF(k=30).fit(train)
summary_a,rows_a,evidence_a=neighbor_view(app_model,data.movies,1,10,False,0.0,5)
summary_b,rows_b,evidence_b=neighbor_view(app_model,data.movies,1,10,True,0.0,5)
display(rows_a)
display(rows_b)


### E5 · 점수 변화의 근거를 읽기
같은 사용자1·k10·N5에서 두 모드의 추천 영화/점수/실제 이웃 수를 비교하라. 한 영화를 공통으로 정해 두 모드의 근거 표도 비교하라. “평균 보정이면 언제나 점수가 낮아진다”는 주장에 동의하는지 근거와 함께 적어라.


In [ ]:
comparison_note=''  # 영화ID, 두 예측값, 대상 평균, 편차 집계와 판단을 기록
print(comparison_note or 'E5: 실제 출력의 근거를 기록하세요.')


### E6 · 화면 입력을 callback에 연결하기
Gradio 버튼의 입력 순서는 `[user,k,centered,threshold,n]`이다. 이를 받는 함수에서 `neighbor_view`의 인자에 명명 인자로 연결하라. k와 n을 바꿔 전달하면 무엇이 달라지는지도 설명하라. 문자열/슬라이더 값을 정수·실수·bool로 바꾸는 경계 처리를 포함한다.


In [ ]:
def my_callback(user,k,centered,threshold,n):
    return None  # (summary, rows, evidence)를 반환하도록 완성
my_result=my_callback(1,10,True,0.0,5)
if my_result is None:
    print('E6: 화면 입력과 callback 인자를 연결하세요.')
else:
    assert len(my_result)==3 and len(my_result[1])<=5


### 기존 앱에 새 탭을 붙여 실행
`FourMethodRecommender.fit(train,users,movies)`는 이전 네 방법 비교 모델이다. `build_cf_app`에 전달해 기본 CF와 기존 네 방법 탭을 만들고, 이를 `build_neighbor_app(...,legacy=...)`에 포함한다. build 함수는 서버를 띄우지 않는다. 마지막 launch가 Colab 공유 주소를 만든다. 런타임 종료 후에는 다시 실행해야 한다.


In [ ]:
comparison=FourMethodRecommender().fit(train,data.users,data.movies)
legacy=build_cf_app(UserCF().fit(train),data.movies,comparison=comparison)
app=build_neighbor_app(app_model,data.movies,legacy=legacy)
if IN_COLAB:
    app.launch(share=True)
else:
    print('로컬 검증: 앱 구성과 callback을 검사합니다. Colab에서는 위 셀이 앱을 실행합니다.')


## 6. 실행 점검
다음은 빈칸 정답을 대신하지 않는 데이터·API·앱의 기본 검사다. 관측 분할, 유한한 예측, 근거 수, 실제 callback 결과를 확인한다. 이 검사는 Colab 웹 화면 자체를 확인한 것과 구별한다.


In [ ]:
assert len(train)+len(validation)+len(test)==len(data.ratings)
assert not pair_set(train)&pair_set(validation) and not pair_set(outer.train)&pair_set(test)
if DATA_MODE=='real':
    assert prepared.mode=='movielens'
    assert (len(data.users),len(data.movies),len(data.ratings))==(943,1682,100000)
    assert (len(train),len(validation),len(test))==(60000,20000,20000)
assert len(scores)==16 and np.isfinite(scores.rmse).all()
assert np.isfinite(final_details.prediction).all() and final_details.prediction.between(1,5).all()
assert np.isfinite(final_rmse) and final_rmse>0
assert not rows_a.empty and not rows_b.empty and not evidence_a.empty and not evidence_b.empty
assert set(rows_a.movie_id).isdisjoint(app_model.seen_[1])
assert rows_a.n_contributors.le(10).all() and rows_b.n_contributors.le(10).all()
assert app_model.k==30 and not app_model.centered
assert any(d.get('api_name')=='neighbor_recommend' for d in app.config['dependencies'])
assert any(d.get('api_name')=='cf_recommend' for d in app.config['dependencies'])
if not IN_COLAB: app.close()
print('데이터 분할·예측·callback·기존 탭 보존 검사 통과')


## 점검 퀴즈
1. 이웃 수 k와 추천 수 N은 각각 무엇을 제한하는가?
2. 목표 영화의 미평가자를 제거하는 일과 상위 k명 선택 중 무엇을 먼저 해야 하는가?
3. KNN과 Thresholding은 이웃 수와 유사도 중 무엇을 직접 통제하는가?
4. 검증 RMSE가 최소인 k를 찾을 때 테스트 데이터를 반복 사용하면 어떤 문제가 생기는가?
5. 사용자 평균을 구할 때 미관측 평점을 0으로 채운 표를 사용하면 안 되는 이유는 무엇인가?
6. 평점편차 기반 CF에서 이웃 평균을 빼고 마지막에 대상 사용자 평균을 더하는 이유는 무엇인가?
7. 평균 보정 후 예측이 5를 넘거나 유효 이웃이 없으면 어떻게 해석해야 하는가?

[학생용 점검 퀴즈 해설](https://github.com/lunalab-ai/recommender/blob/2026-fall-w04a/course/handouts/w04a-neighborhood-cf-quiz.md)

## 가져갈 내용과 참고자료
목표 영화별로 이웃을 고른다. 설정은 검증으로 선택한다. 이웃 평균을 빼고 대상 평균을 더한다.
주교재3.4–3.6의 용어·개념을 따른 독립 구현이며, 검증/테스트 분리와 근거 표시는 수업 보충이다. 원본 스캔·출판사 코드는 포함하지 않는다.
[Surprise k-NN 수식](https://surprise.readthedocs.io/en/stable/knn_inspired.html) · [scikit-learn 검증 안내](https://scikit-learn.org/stable/modules/cross_validation.html) · [MovieLens100K 공식 배포](https://grouplens.org/datasets/movielens/100k/)
